# Web Development & API Design
This notebook covers REST architecture, Flask API design, authentication, deployment options, Dockerization, and GPU-enabled container usage on Google Colab.

## **1. REST Architecture Principles**

**Concepts:**
- Representational State Transfer (REST) is an architectural style for designing networked applications.
- Key principles:
  1. **Statelessness**: Each request contains all the information needed to process it.
  2. **Client-Server Separation**: The client and server can evolve independently.
  3. **Cacheability**: Responses should indicate if they are cacheable.
  4. **Uniform Interface**: Standardized operations (GET, POST, PUT, DELETE).
  5. **Layered System**: Architecture can have intermediaries such as proxies or load balancers.

**Example:**
- `GET /users` → returns all users
- `POST /users` → creates a new user



## **2. Building APIs with Flask**

Python Flask is a widely used framework for developing web applications and APIs in Python. It enables developers to create **RESTful APIs** quickly and efficiently, which can then be utilised by other software applications.

**Flask Installation:**
```bash
pip install Flask
```

In [ ]:
pip install Flask

Check this and read carefully **before** running the next two cells:
1. Navigate to **Week5 directory**.
2. Open  [```app.py```](app.py).
3. Review the code well.
4. Afterwards, go back to **Week5 directory** and open in **terminal**.
5. Type ```pip install flask```.
6. Then, type ```python app.py```.
7. Follow this [link](http://127.0.0.1:5000/users). Intially, there are no users, thus it displays ```[]```.
8. Scan the code in the cell below and run it afterwards.
9. Then refresh the link and apply the same steps (7-8) for the upcoming cell as well.
10. Once you are done and understood the code and the output, please go back to the terminal and press ```CTRL+C``` to quit.

In [1]:
# The 'requests' library simplifies making HTTP requests (GET, POST, PUT, DELETE, etc.)
# It handles connection management, headers, and response parsing under the hood.
import requests

# The base URL pointing to our locally running Flask server.
# 127.0.0.1 is the loopback address (localhost). **Only** accessible on this machine.
# 5000 is Flask's default development port.
url = "http://127.0.0.1:5000/users"

# The user payload we want to send a plain Python dict.
# requests will automatically serialize this to a JSON string when json= is used below.
user = {"name": "Adam", "email": "adam@example.com"}

# --- POST: Create a new user ---
# requests.post() sends an HTTP POST request to the Flask /users endpoint.
# The json=user argument does two things automatically:
#   1. Serializes the dict to a JSON string body
#   2. Sets the Content-Type header to application/json (so Flask knows how to parse it)
response = requests.post(url, json=user)

# .json() deserializes the JSON response body back into a Python dict.
# Here we expect: {"message": "User added"}
print(response.json())

# --- GET: Fetch all users ---
# requests.get() sends an HTTP GET request to retrieve the full users list.
# .json() is chained directly since we only care about the response body.
# Here we expect: [{"name": "Adam", "email": "adam@example.com"}]
all_users = requests.get(url).json()
print(all_users)

{'message': 'User added'}
[{'email': 'adam@example.com', 'name': 'Adam'}]


In [ ]:
# Flask: web framework for building the API
# threading: allows running Flask server concurrently alongside the main script
# requests: HTTP client for sending requests to the Flask server
# time: used to pause execution while the Flask server initializes
from flask import Flask, request, jsonify
import threading
import requests
import time

# Create the Flask app instance with a user-defined name.
# Unlike __name__, a string name like this is useful for clarity in logs and debugging
# when the app isn't being run as a standalone module.
app = Flask("Adding New Users")

# In-memory list that will hold every user added via POST.
# Without this, add_user() below would raise a NameError.
users = []


# Route: GET /users
# Returns the current list of all users serialized as a JSON array.
@app.route("/users", methods=["GET"])
def get_users():
    return jsonify(users)  # return the full list, not the single `user` dict

# Route: POST /users
# Accepts a JSON body, appends the parsed dict to the users list,
# and responds with a 201 Created status to confirm the resource was added.
@app.route("/users", methods=["POST"])
def add_user():
    data = request.get_json()  # Parse JSON body from the incoming POST request
    users.append(data)         # Add the new user to the shared in-memory list
    return jsonify({"message": "User added"}), 201

# Wraps app.run() in a plain function so it can be passed as a target to Thread().
# - debug=False: required in threaded contexts — debug mode spawns a child process
#   which conflicts with threading and causes the server to fail or behave unexpectedly.
# - use_reloader=False: disables Werkzeug's file-watcher reloader, which also
#   spawns a subprocess and is incompatible with threading.
def run_flask():
    app.run(debug=False, use_reloader=False)

# Launch the Flask server in a background (daemon) thread.
# - target=run_flask: the function the thread will execute
# - daemon=True: marks the thread as a daemon, meaning it will automatically shut down
#   when the main script exits, preventing the program from hanging indefinitely.
threading.Thread(target=run_flask, daemon=True).start()

# Pause the main thread for 1 second to give Flask time to bind to the port
# and begin listening for connections before we send requests.
# Without this, requests sent immediately may fail with a "Connection refused" error.
time.sleep(1)

# --- Send HTTP requests to the now-running Flask server ---

url = "http://127.0.0.1:5000/users"
user = {"name": "Mona", "email": "Mona@example.com"}

# POST: Send Mona's data as JSON to the /users endpoint and print the server's response.
# Expected output: {"message": "User added"}
print(requests.post(url, json=user).json())

# GET: Retrieve and print the full users list from the server.
# Expected output: [{"name": "Mona", "email": "Mona@example.com"}]
print(requests.get(url).json())

 * Serving Flask app 'Adding New Users'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


{'message': 'User added'}
[{'email': 'adam@example.com', 'name': 'Adam'}, {'email': 'Mona@example.com', 'name': 'Mona'}]


: 

## **3. API Structure and Routing**

When a request hits the server, Flask checks its **routing table** and calls the matching function and this is called **dispatching**. 
The core principle of API structuring is as follow:  routes are defined modularly and assembled centrally, which keeps the codebase maintainable as the API grows.

### HTTP Methods
Each method signals **intent**:

| Method | Intent | Example |
|---|---|---|
| `GET` | Read data | Fetch all users |
| `POST` | Create data | Add a new user |
| `PUT` | Replace data | Overwrite a user |
| `PATCH` | Update data | Edit a user's email |
| `DELETE` | Remove data | Delete a user |

**Recommended Structure: (For illustrative purposes ONLY)**
```
project/
│
├─ app.py              # main Flask app
├─ routes/
│   ├─ user_routes.py  # blueprint definition
│   └─ model_routes.py
├─ models/
│   └─ my_model.py
├─ requirements.txt
└─ Dockerfile
```
**General Comments**:

- In the recommended structure,  ```user_routes.py```  defines which URLs exist and what functions handle them.
 Using a Blueprint instead of registering **directly** on app makes this a *self-contained, reusable routing module*. It has no knowledge of the broader app. It just defines its own routes and waits to be mounted.

- ```app.py``` is where the structure is defined. It imports **blueprints** and assigns them URL prefixes, establishing the API's overall URL hierarchy. It acts as a central registry, wiring together all the independent routing modules into **one cohesive application**.
---
Have a look at the code of API structure and Routing:
1. Navigate to **project directory in Week5 directory**.
2. Go to **routes directory** and check the code of ```user_routes.py```.
3. Then, check the code in ```app.py```.

**Questions:**  [Don't forget to answer these reflection questions]
- What is the difference between defining a route and registering a route? Which file is responsible for each?
- Why is the route path "/" in user_routes.py but the full URL becomes "/users/"? What controls that transformation?
- Why is it beneficial to keep routes in a separate file from the main app.py?

## **4. Authentication Strategies**
Authentication is the process of verifying who a user is before granting access to protected resources.

**The Golden Rule:**
- Authentication verifies who you are.
- Authorization (the next step) verifies what you are allowed to do.
These **two concepts** are distinct but always work together in a secure API.

There are several authentication strategies such as  Session-Based, API Key, OAuth 2.0, and many more. This week, we will focus on **Token-Based Authentication (JWT)**.

## **Token-Based Authentication:**
The server issues a JSON Web Token after login. The client stores it and sends it with every request. The server validates it **without storing anything**.

### **Instructions:**
1. Open  [```server.py```](server.py) and [```client.py```](client.py).
2. Review the code well.
3. Afterwards, go back to **Week5 directory** and open in **terminal**.
4. Type ```pip install Flask-JWT-Extended```.
5. Then, type ```python server.py```.
6. Open a **new terminal**  and type ```python client.py```. Note: if you encounter an error, make sure that the port is free --> ```netstat -ano | findstr :5000```. If it returns two separate processes that bound to port 5000, then this confirms the conflict. Kill both, then verify the port is actually clear before restarting anything ```taskkill /PID <that_pid> /F```.
6. Check the output and go back to [```server.py```](server.py) and [```client.py```](client.py). Then, answer the below questions.
---
### **Here is workflow overview**:
```
Client                          Server
  |                               |
  |-- POST /login (credentials) ->|  Validates user, generates JWT
  |<- 200 OK { access_token } ----|
  |                               |
  |-- GET /protected (JWT) ------>|  Verifies JWT signature
  |<- 200 OK { "Access granted" }-|
```

**Step 1 : Identity verification**:
Client sends credentials → Server validates → Server issues signed JWT

**Step 2 : Authorized access**:
Client sends JWT in header → Server validates signature → Server grants access

**Running order:**
```
1. python server.py   ← start this first
2. python client.py   ← run this after server is running
```

**Questions: [Don't forget to answer these reflection questions]** 
- What is the purpose of the ```/login``` route?  What goes in, and what comes out?
- What is the purpose of the ```/protected``` route? Who can access it and who can't?
- What is the role of JWTManager(app)? What would break if you removed that line?

## **5. Deployment Options:**

Here are the main deployment options for this Flask JWT application:

- **Local deployment:** ```python app.py```

- **Production deployment:** using ```Gunicorn```, ```Waitress``` , ```uWSGI```, Dockers, etc. 

- **Cloud deployment:** AWS, Google Cloud, Heroku, etc.

> Now we focus on Web Server Gateway Interface (WSGI) such as ```Gunicorn``` for Mac/Linux users and  ```Waitress``` for Windows users.

### Instructions for **Mac/Linux users**

1. Check the simple code in   [```production.py```](production.py). 
2. Go to **Week5 directory** and open in **terminal**. 
3. Type ```pip install gunicorn```.
4. Type ```gunicorn production:app```.
> gunicorn production:app starts a production-ready server on Linux/Mac and tells it to run the app object from production.py. Same job as the Waitress command for windows users, just the Linux/Mac equivalent. Note: it defaults to port 8000, not 5000.
5. Navigate to this [link](http://127.0.0.1:8000/) and compare the desired output in [```production.py```](production.py).

### Instructions for **Windows users**

1. Check the simple code in   [```production.py```](production.py). 
2. Go to **Week5 directory** and open in **terminal**. 
3. Type ```pip install waitress```.
4. Type ```python -m waitress --listen=0.0.0.0:5000 production:app```. 
> Waitress is a production-grade WSGI server; this command tells it to listen on port 5000 across all network interfaces (0.0.0.0) and serve the app object it finds inside production.py. Unlike development (python app.py, which calls app.run() itself), production hands that job to Waitress. Your Flask file just needs to expose app for it to import and run.
5. Navigate to this [link](http://127.0.0.1:5000/) and compare the desired output in [```production.py```](production.py).

### **Questions** [Don't forget to answer these reflection questions]
- What is the purpose of the ```app``` object in production.py, and why must ```Gunicorn``` or ```Waitress``` reference it explicitly (production:app)?
- What would happen if you accidentally ran python ```production.py``` in production instead of using a ```WSGI``` server?

## **6. Docker:**

Docker is a platform that packages an application and all its dependencies into a single portable unit called a container. The key benefit is consistency as the app runs identically on any machine.

**Key Concepts:**
| Term           | Description                                                                 |
|----------------|-----------------------------------------------------------------------------|
|**Docker Image**   | A blueprint of an application, containing code, dependencies, and configurations. |
| **Docker Container** | A running instance of a Docker image.                                     |
| **Dockerfile**     | A script containing instructions to build a Docker image.                  |
| **Docker Hub**     | A repository for storing and sharing Docker images.                        |

### Installation & Verification (SKIP these steps if you have already done the pre-preparation)
- Install [Docker Desktop](https://www.docker.com/products/docker-desktop/). Run the installer and follow the instructions.
- Make sure to enable **virtualization** and verify installation. 
- Open terminal and run : ```docker --version```. You should see something like ```Docker version 29.2.1, build a5c7197```.
- Start Docker Desktop and make sure that Docker Desktop is running.

### General Docker Commands (Check them well)
| Command | Description |
|---------|-------------|
| `docker ps` | Lists all running containers. Add `-a` to see all containers, including stopped ones. |
| `docker images` | Shows all Docker images available locally, including tags, IDs, and sizes. |
| `docker logs <container_id_or_name>` | Displays the stdout and stderr logs of a specific container for debugging or monitoring. |
| `docker stop <container_id_or_name>` | Gracefully stops a running container by sending a termination signal. |
| `docker rm <container_id_or_name>` | Removes a stopped container from the system to free resources. Containers must be stopped before removal. |

### 1. Setup (Dockerise your Flask App)
Deploying a Flask application manually can result in dependency issues and inconsistencies across different environments. Docker addresses this problem by packaging applications into containers, ensuring that they run consistently on any system.

Check  the codes in **Week5 directory** and we focus on the following:
- [```docker.py```](docker.py): Contains the Flask application code, defining routes and app behavior.
- [```Dockerfile.dev```](Dockerfile.dev): Specifies instructions for building a Docker image, including the base Python image, required files, dependencies, and the command to run the application. Dockerfile has **no extension**. 
- [```requirements.txt```](requirements.txt): Lists the Python libraries needed for the app (like Flask) so Docker can install them during the build process.

### Building and Running the Docker Container

- Build your container: Open terminal and run: ```docker buildx build -t flask-api-dev -f Dockerfile.dev .``` 
> builds a Docker image named ```flask-api-dev``` using the instructions in [```Dockerfile.dev```](Dockerfile.dev). The trailing ```.``` sets the current directory as the build context --> the source of any files the Dockerfile copies in.
- Run your container: then run : ```docker run -p 5000:5000 flask-api-dev```
> starts a container from the flask-api-dev image you just built. The -p 5000:5000 flag maps port 5000 on your machine to port 5000 inside the container, so requests to http://localhost:5000 on your host actually reach the Flask app running inside it.
- Navigate to [Link](http://localhost:5000/) or in the terminal ```curl http://localhost:5000/```
- Then Navigate to [Users Link](http://localhost:5000/users) or in the terminal ```curl http://localhost:5000/users```
- Close the server by pressing ```CTRL + C```

### 2. Setup (Dockerise your **Production** Flask App)
- Dockerising a production Flask application using ```Waitress``` or ```Gunicorn``` allows the app to run in a consistent and isolated environment, ensuring reliable behaviour across development, staging, and production. 
- Both ```Waitress``` and ```Gunicorn``` provide production-grade WSGI servers capable of handling multiple concurrent requests, unlike Flask’s built-in development server. 
- Using Docker with these servers also simplifies deployment, portability, and scalability, making it easier to manage and run the application on different systems or cloud platforms.

Check  the codes in **Week5 directory** and we focus on the following:
- [```run_prod.py```](run_prod.py): Contains the Flask application code, defining routes and app behavior.
- [```Dockerfile.prod```](Dockerfile.prod): Specifies instructions for building a Docker image, including the base Python image, required files, dependencies, and the command to run the application. Dockerfile has **no extension**. 
- [```requirements.txt```](requirements.txt): Lists the Python libraries needed for the app (like Flask) so Docker can install them during the build process.

### Building and Running the Docker Container

- Build your container: Open terminal and run: ```docker buildx build -t flask-api-prod -f Dockerfile.prod .``` 
- Run your container: then run : ```docker run -p 8080:8080 flask-api-prod```
- Navigate to [Link]( http://localhost:8080/) or in the terminal ```curl http://localhost:8080/```
- Then Navigate to [Users Link]( http://localhost:8080/users) or in the terminal ```curl http://localhost:8080/users```
- Close the server by pressing ```CTRL + C```

### Questions [Don't forget to answer these reflection questions]

- How does Docker ensure that a Flask app runs consistently across different machines and environments?  
- What are the benefits of using Docker containers for deploying a production Flask app compared with running it directly on the host system?  
- How does port mapping in Docker affect how you access a Flask application from your host machine?  

## **7. GPU-enabled container concepts:**

Navigate to Google Colab and upload [GPU enabled Flask API](GPU_enabled_Flask_API.ipynb)

### Now we are done with week 5's lab content 🎉🎉🎉 ###
### Let's navigate to Exercise's notebook
[Exercise Notebook](Week5_Exercise.ipynb)